# Waste Classification CNN — Training on Google Colab (GPU)

This notebook trains the models from the [GitHub repo](https://github.com/Utsav-tala/waste-classification-cnn) on a free Colab GPU.
Code is written locally and pushed to GitHub; **only training happens here**.

**Before you start:** `Runtime → Change runtime type → T4 GPU → Save`.

Workflow: clone repo → install pinned TensorFlow → download TrashNet → verify data → train → save to Google Drive → download results → commit them locally.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Clone (or update) the repository

In [ ]:
import os
REPO_DIR = "/content/waste-classification-cnn"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Utsav-tala/waste-classification-cnn.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
%cd {REPO_DIR}
!git log --oneline -3

## 3. Install the same TensorFlow/Keras versions as the local machine

Matching versions means the trained `.keras` files load in the local Streamlit app.
If Colab shows **"Restart session"** after this cell, click it, then continue from step 4 (don't re-run this cell).

In [ ]:
!pip install -q "tensorflow[and-cuda]==2.21.0" "keras==3.15.1"

## 4. Confirm versions and GPU

In [ ]:
%cd /content/waste-classification-cnn
import tensorflow as tf, keras
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
assert gpus, "No GPU found: Runtime → Change runtime type → T4 GPU"

## 5. Download TrashNet and verify it matches our splits

Downloads to Colab's local disk (fast), then checks every file's SHA-256 against `data/splits/*.csv`,
so we know the images are exactly the ones the splits were made from.

In [ ]:
%cd /content/waste-classification-cnn
if not os.path.exists("dataset-resized"):
    !wget -q https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip -O /content/dataset-resized.zip
    !unzip -q /content/dataset-resized.zip -d /content/waste-classification-cnn
    !rm -rf /content/waste-classification-cnn/__MACOSX

In [ ]:
import hashlib
import pandas as pd

splits = pd.concat([pd.read_csv(f"data/splits/{s}.csv") for s in ["train", "val", "test"]])
missing, mismatched = 0, 0
for fp, expected in zip(splits["filepath"], splits["sha256"]):
    if not os.path.exists(fp):
        missing += 1
        continue
    if hashlib.sha256(open(fp, "rb").read()).hexdigest() != expected:
        mismatched += 1
print(f"Images in splits: {len(splits)} | missing: {missing} | hash mismatch: {mismatched}")
assert missing == 0 and mismatched == 0, "Dataset does not match the split files"

## 6. Mount Google Drive (results survive if the session disconnects)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/waste-classification-cnn/runs"
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Results will be copied to:", DRIVE_DIR)

## 7. Quick smoke test of all architectures

In [ ]:
%cd /content/waste-classification-cnn
!python -m src.model

## 8. Train

Choose the models and recipe. Run names: `<model>` for recipe v1, `<model>_<recipe>` otherwise (e.g. `model_c_v3`).

Recipe **v3** trains for the full 100 epochs (≈10 min for Model C on a T4) and keeps the final weights; compare runs on `saved_val_accuracy`.

| Option | Models | Purpose |
|---|---|---|
| Main | `model_c` | New residual CNN |
| Ablation | `model_b model_c_plain model_c` | Effect of recipe, depth and skip connections |

In [ ]:
MODELS = "model_c"   # e.g. "model_b model_c_plain model_c" for the full ablation
RECIPE = "v3"   # v3 = full cosine run, no early stopping, BN momentum 0.9

%cd /content/waste-classification-cnn
!python -m src.train --models {MODELS} --recipe {RECIPE}

## 9. Copy results to Google Drive

In [ ]:
import shutil, datetime
stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
dest = f"{DRIVE_DIR}/{stamp}"
shutil.copytree("models", f"{dest}/models")
shutil.copytree("results", f"{dest}/results")
print("Copied to", dest)

## 10. (Final step only) Evaluate the chosen model on the test set

Use the test set **once**, after choosing the final model on validation accuracy.
Then re-run step 9 so the test results are saved too.

In [ ]:
RUN_NAME = "model_c_v3"
# !python -m src.evaluate {RUN_NAME}

## 11. Download results to your computer

Unzip into the local project folder (it overwrites `models/` and `results/` with the new files), check the results, then commit and push from your machine.

In [ ]:
!zip -qr /content/colab_results.zip models results
from google.colab import files
files.download("/content/colab_results.zip")